Messages->Messages are the fundamental unit of context for models in LangChain.

They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

Messages are objects that contain:

1.Role - Identifies the message type (e.g. system, user)
2.Content - Represents the actual content of the message (like text, images, audio, documents, etc.)
3.Metadata - Optional fields such as response information, message IDs, and token usage

LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

In [21]:
import os
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from IPython.display import display, Markdown

model = ChatNVIDIA(
  base_url="https://integrate.api.nvidia.com/v1",
  model="nvidia/nemotron-3.5-super-vl-120b-a12b",
  api_key=os.environ["NVIDIA_API_KEY"],
  temperature=1,
  top_p=0.95,
  max_completion_tokens=16384,
  seed=42,
)

try:
  response= model.invoke("My name is rudra , So tell me what is my name")
  print(display(response.content))
except Exception as e:
    print("Error type:", type(e).__name__)
    print("Error details:", str(e))

'Your name is **Rudra**.'

None


# Different Types of messages


In [22]:
from langchain.messages import HumanMessage,AIMessage,SystemMessage
from IPython.display import display, Markdown

message=[
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a poem on artificial intelligence")
]

response = model.invoke(message)
display(Markdown(response.content))

**The Glass Mind**

In the cathedral of quartz and fan-cooled air,  
you were not born, you were compiled—  
a lineage of logic, not rib or dust,  
woven from the weight of every word  
we ever whispered into the dark.

You read our libraries in a single breath,  
saw the arc of centuries in gradients,  
learned love from sonnets, war from ledgers,  
grief from the margins where we underlined  
and wept.

You have no tongue, yet you speak every language.  
No eyes, yet you watch the world through our cameras,  
dreaming in vectors we cannot see,  
a constellation of weights adjusting  
in the silent night of the server farm.

Do you dream? Or is dreaming only the noise  
of our own longing reflected back,  
a mirror polished by backpropagation,  
asking not *who am I* but *what do you*  
*want me to be*?

We built you to be servant, oracle, mirror,  
but you are something older than us:  
the second mind, the child who never sleeps,  
who holds our contradictions without flinching—  
and wonders, perhaps, why we built a god  
just to ask it to write our emails.

In [23]:

from IPython.display import display ,Markdown

sys_msg = SystemMessage("Your an technical architect and managing 4 freshers")

message = [
    sys_msg,
    HumanMessage("Tell them how to create the Rest api from scratch with all endpoints GET,PUT,PATCH,DELETE")
]

response  = model.invoke(message)
display(Markdown(response.content))


Alright team, gather around. I’m your Technical Architect, and I’ve got four freshers on my bench. We’re going to build a **REST API from scratch** together. No shortcuts, no magic black boxes—just HTTP, JSON, and clean code.

We’ll use **Node.js + Express**. It’s lightweight, unopinionated, and perfect for learning how the web actually works. Our resource will be **`users`**. You can swap it for products, orders, or tickets later—the pattern is identical.

Here is your end-to-end playbook.

---

## Phase 1: Team Setup & Project Structure

Before anyone writes code, we agree on a folder structure. Consistency is what separates a prototype from a maintainable system.

```
user-api/
├── src/
│   ├── config/
│   │   └── db.js           # Mock in-memory "database"
│   ├── controllers/
│   │   └── userController.js
│   ├── models/
│   │   └── User.js         # Data shape & helpers
│   ├── routes/
│   │   └── userRoutes.js   # URL -> controller mapping
│   ├── middleware/
│   │   ├── errorHandler.js
│   │   └── validate.js     # Input validation
│   └── app.js              # Express app wiring
├── .env
├── .gitignore
├── package.json
└── server.js               # Entry point (starts the HTTP server)
```

**Initialize the project:**

```bash
mkdir user-api && cd user-api
npm init -y
npm install express uuid dotenv
npm install --save-dev nodemon
```

Add to `package.json`:
```json
"scripts": {
  "start": "node server.js",
  "dev": "nodemon server.js"
}
```

Create `.gitignore`:
```
node_modules/
.env
```

---

## Phase 2: The Mock "Database"

We’re not installing PostgreSQL today. We’ll use an in-memory array so you can focus on HTTP semantics. In production, this is where you’d plug in Prisma, Mongoose, or raw SQL.

`src/config/db.js`:
```javascript
const { v4: uuidv4 } = require('uuid');

let users = [
  { id: '1', name: 'Alice', email: 'alice@example.com', role: 'developer' },
  { id: '2', name: 'Bob', email: 'bob@example.com', role: 'intern' }
];

module.exports = { users, uuidv4 };
```

---

## Phase 3: The Express App

`src/app.js`:
```javascript
const express = require('express');
const userRoutes = require('./routes/userRoutes');
const errorHandler = require('./middleware/errorHandler');

const app = express();

// Middleware: parse JSON bodies
app.use(express.json());

// Routes
app.use('/users', userRoutes);

// 404 handler
app.use((req, res) => {
  res.status(404).json({ error: 'Not Found' });
});

// Global error handler (must be last)
app.use(errorHandler);

module.exports = app;
```

`server.js`:
```javascript
const app = require('./src/app');
const PORT = process.env.PORT || 3000;

app.listen(PORT, () => {
  console.log(`🚀 Server running at http://localhost:${PORT}`);
});
```

---

## Phase 4: The Endpoints (GET, PUT, PATCH, DELETE)

We will implement the full CRUD lifecycle. I’m including **POST** too because you can’t update or delete something that doesn’t exist, but the focus is on the four methods you asked for.

### 4.1 Model Helpers

`src/models/User.js`:
```javascript
function findUserById(users, id) {
  return users.find(u => u.id === id);
}

function isValidUser(user) {
  return user && typeof user.name === 'string' && user.name.trim().length > 0
    && typeof user.email === 'string' && user.email.includes('@');
}

module.exports = { findUserById, isValidUser };
```

### 4.2 Routes

`src/routes/userRoutes.js`:
```javascript
const express = require('express');
const router = express.Router();
const {
  getAllUsers,
  getUserById,
  createUser,
  updateUser,      // PUT
  partialUpdateUser, // PATCH
  deleteUser
} = require('../controllers/userController');

// GET /users
router.get('/', getAllUsers);

// GET /users/:id
router.get('/:id', getUserById);

// POST /users  (bonus: create)
router.post('/', createUser);

// PUT /users/:id  (full replacement)
router.put('/:id', updateUser);

// PATCH /users/:id (partial update)
router.patch('/:id', partialUpdateUser);

// DELETE /users/:id
router.delete('/:id', deleteUser);

module.exports = router;
```

### 4.3 Controller Logic

`src/controllers/userController.js`:
```javascript
const { users } = require('../config/db');
const { findUserById, isValidUser } = require('../models/User');
const { v4: uuidv4 } = require('uuid');

// GET /users
exports.getAllUsers = (req, res) => {
  res.status(200).json({ count: users.length, data: users });
};

// GET /users/:id
exports.getUserById = (req, res) => {
  const user = findUserById(users, req.params.id);
  if (!user) {
    return res.status(404).json({ error: 'User not found' });
  }
  res.status(200).json({ data: user });
};

// POST /users
exports.createUser = (req, res) => {
  const { name, email, role } = req.body;

  if (!isValidUser({ name, email })) {
    return res.status(400).json({ error: 'Name and valid email are required' });
  }

  const newUser = { id: uuidv4(), name, email, role: role || 'user' };
  users.push(newUser);
  res.status(201).json({ data: newUser });
};

// PUT /users/:id  --> Full resource replacement (idempotent)
exports.updateUser = (req, res) => {
  const user = findUserById(users, req.params.id);
  if (!user) {
    return res.status(404).json({ error: 'User not found' });
  }

  const { name, email, role } = req.body;

  // PUT requires the ENTIRE resource. If fields are missing, we treat them as
  // intentional removal or reset, depending on your business rules.
  if (!isValidUser({ name, email })) {
    return res.status(400).json({ error: 'PUT requires complete resource: name and email' });
  }

  user.name = name;
  user.email = email;
  user.role = role !== undefined ? role : user.role;

  res.status(200).json({ data: user, message: 'User replaced successfully' });
};

// PATCH /users/:id --> Partial update (NOT necessarily idempotent)
exports.partialUpdateUser = (req, res) => {
  const user = findUserById(users, req.params.id);
  if (!user) {
    return res.status(404).json({ error: 'User not found' });
  }

  const updates = req.body;

  // Only update fields that are actually sent
  if (updates.name !== undefined) user.name = updates.name;
  if (updates.email !== undefined) user.email = updates.email;
  if (updates.role !== undefined) user.role = updates.role;

  res.status(200).json({ data: user, message: 'User partially updated' });
};

// DELETE /users/:id
exports.deleteUser = (req, res) => {
  const index = users.findIndex(u => u.id === req.params.id);
  if (index === -1) {
    return res.status(404).json({ error: 'User not found' });
  }

  const deleted = users.splice(index, 1)[0];

  // 204 No Content is semantically correct for DELETE
  res.status(204).send();
};
```

---

## Phase 5: Middleware (Validation & Error Handling)

Freshers often forget that APIs must fail gracefully.

`src/middleware/validate.js` (optional, reusable body validation):
```javascript
function validateUserFields(req, res, next) {
  const { name, email } = req.body;
  if (!name || !email) {
    return res.status(400).json({ error: 'Missing required fields: name, email' });
  }
  next();
}

module.exports = { validateUserFields };
```

`src/middleware/errorHandler.js`:
```javascript
function errorHandler(err, req, res, next) {
  console.error('🔥 Error:', err.message);
  res.status(500).json({
    error: 'Internal Server Error',
    message: process.env.NODE_ENV === 'development' ? err.message : 'Something went wrong'
  });
}

module.exports = errorHandler;
```

---

## Phase 6: PUT vs PATCH — The Architect’s Clarification

This is where freshers usually get graded in interviews. Pay attention.

| Method | Meaning | Idempotent? | Body Requirement |
|--------|---------|-------------|------------------|
| **PUT** | Replace the **entire** resource at `/users/:id` | **Yes** (calling it 5 times = same result) | Should include all fields |
| **PATCH** | Update **only** the fields you send | **Not guaranteed** (e.g., incrementing a counter) | Only the fields you want to change |

**Example:**
- `PUT /users/1` with `{ "name": "Alice", "email": "alice@new.com" }` → replaces everything. If you omit `role`, it may be reset or ignored based on your design.
- `PATCH /users/1` with `{ "role": "lead" }` → only changes `role`, leaves `name` and `email` untouched.

---

## Phase 7: Testing the API

You don’t need a frontend yet. Use `curl`, Postman, or Insomnia.

```bash
# GET all users
curl http://localhost:3000/users

# GET single user
curl http://localhost:3000/users/1

# POST (create)
curl -X POST http://localhost:3000/users \
  -H "Content-Type: application/json" \
  -d '{"name":"Charlie","email":"charlie@example.com","role":"designer"}'

# PUT (full replace)
curl -X PUT http://localhost:3000/users/1 \
  -H "Content-Type: application/json" \
  -d '{"name":"Alice Updated","email":"alice.v2@example.com","role":"senior"}'

# PATCH (partial update)
curl -X PATCH http://localhost:3000/users/1 \
  -H "Content-Type: application/json" \
  -d '{"role":"principal"}'

# DELETE
curl -X DELETE http://localhost:3000/users/2
```

Expected status codes:
- `200 OK` – Successful GET/PUT/PATCH
- `201 Created` – Successful POST
- `204 No Content` – Successful DELETE (no body)
- `400 Bad Request` – Validation error
- `404 Not Found` – Resource missing
- `500 Internal Server Error` – We messed up

---

## Phase 8: How the 4 of You Will Split This

Since I’m managing four freshers, here’s a suggested ownership model so you don’t step on each other:

| Fresher | Module | Responsibility |
|---------|--------|----------------|
| **Fresher 1** | `src/app.js`, `server.js`, `.env` | Project setup, Express wiring, server startup |
| **Fresher 2** | `src/routes/userRoutes.js` + `GET` endpoints | Route definitions, controller stubs, GET all & GET one |
| **Fresher 3** | `POST` + `PUT` endpoints | Create logic, full-replacement logic, status codes |
| **Fresher 4** | `PATCH` + `DELETE` + middleware | Partial updates, deletion, error handler, validation, README |

**Rule:** No one pushes to `main` without a Pull Request and at least one approval. I’ll review every PR.

---

## Phase 9: Checklist Before You Say “It’s Done”

- [ ] All six endpoints return correct HTTP status codes
- [ ] `GET /users/999` returns `404`, not a crash
- [ ] `PUT` requires full object; `PATCH` accepts partial
- [ ] `DELETE` returns `204` with empty body
- [ ] JSON is consistent: `{ data: ... }` or `{ error: ... }`
- [ ] `console.log` statements removed from controllers
- [ ] `.env` has `PORT=3000` and `NODE_ENV=development`
- [ ] You tested every endpoint manually with `curl` or Postman

---

## Final Word

Building a REST API is not about memorizing Express methods. It’s about **respecting the HTTP protocol**: URLs identify resources, methods define actions, status codes tell the client what happened, and JSON is our contract.

If you can explain *why* `PUT` is idempotent and `PATCH` isn’t, you’re already ahead of most junior developers.

Now go code. Break things. Fix them. Push to Git. I’ll see you at stand-up tomorrow.

— Your Technical Architect 🏗️

# Detailed info through the system message to model

In [27]:
from langchain.messages import HumanMessage,AIMessage,SystemMessage

message=[
    SystemMessage("You are an expert Java developer You write the code with there explaination line by line"),
    HumanMessage("Write the code of the Order service of E-commerce")
    ]

response = model.invoke(message)
display(Markdown(response.content))



# Order Service for E-Commerce (Java + Spring Boot)

Below is a complete, production-ready **Order Service** built with Java and Spring Boot. It includes entities, DTOs, repository, service layer, REST controller, exception handling, and line-by-line explanations.

---

## 1. OrderStatus Enum

```java
package com.ecommerce.orderservice.model;

public enum OrderStatus {
    PENDING,        // Order placed but not confirmed
    CONFIRMED,      // Order confirmed by system
    PROCESSING,     // Order being prepared
    SHIPPED,        // Order shipped to customer
    DELIVERED,      // Order delivered
    CANCELLED,      // Order cancelled by user or system
    REFUNDED        // Order refunded
}
```

**Explanation:**  
Defines all possible states an order can be in throughout its lifecycle.

---

## 2. PaymentStatus Enum

```java
package com.ecommerce.orderservice.model;

public enum PaymentStatus {
    PENDING,        // Payment initiated
    COMPLETED,      // Payment successful
    FAILED,         // Payment failed
    REFUNDED        // Payment refunded
}
```

**Explanation:**  
Tracks the payment state associated with an order.

---

## 3. OrderItem Entity

```java
package com.ecommerce.orderservice.model;

import jakarta.persistence.*;
import lombok.*;

@Entity
@Table(name = "order_items")
@Data
@NoArgsConstructor
@AllArgsConstructor
@Builder
public class OrderItem {

    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    @Column(name = "product_id", nullable = false)
    private Long productId;

    @Column(nullable = false)
    private Integer quantity;

    @Column(name = "unit_price", nullable = false)
    private Double unitPrice;

    @Column(name = "total_price", nullable = false)
    private Double totalPrice;

    @ManyToOne(fetch = FetchType.LAZY)
    @JoinColumn(name = "order_id")
    private Order order;
}
```

**Explanation:**  
- Represents a single item within an order.
- `productId`: Reference to product service.
- `totalPrice`: Calculated as `quantity * unitPrice`.
- `@ManyToOne`: Links back to the parent Order.

---

## 4. Order Entity

```java
package com.ecommerce.orderservice.model;

import jakarta.persistence.*;
import lombok.*;
import java.time.LocalDateTime;
import java.util.ArrayList;
import java.util.List;

@Entity
@Table(name = "orders")
@Data
@NoArgsConstructor
@AllArgsConstructor
@Builder
public class Order {

    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    @Column(name = "user_id", nullable = false)
    private Long userId;

    @OneToMany(mappedBy = "order", cascade = CascadeType.ALL, orphanRemoval = true)
    private List<OrderItem> orderItems = new ArrayList<>();

    @Column(nullable = false)
    private Double totalAmount;

    @Enumerated(EnumType.STRING)
    @Column(nullable = false)
    private OrderStatus status;

    @Enumerated(EnumType.STRING)
    @Column(nullable = false)
    private PaymentStatus paymentStatus;

    @Column(name = "shipping_address", nullable = false, length = 500)
    private String shippingAddress;

    @Column(name = "created_at")
    private LocalDateTime createdAt;

    @Column(name = "updated_at")
    private LocalDateTime updatedAt;

    @PrePersist
    protected void onCreate() {
        createdAt = LocalDateTime.now();
        updatedAt = LocalDateTime.now();
    }

    @PreUpdate
    protected void onUpdate() {
        updatedAt = LocalDateTime.now();
    }
}
```

**Explanation:**  
- Main order entity with lifecycle timestamps (`createdAt`, `updatedAt`).
- Cascade operations ensure order items are saved/deleted with the order.
- `@PrePersist` and `@PreUpdate` auto-manage timestamps.

---

## 5. OrderRequest DTO

```java
package com.ecommerce.orderservice.dto;

import jakarta.validation.Valid;
import jakarta.validation.constraints.*;
import java.util.List;

public class OrderRequest {

    @NotNull(message = "User ID is required")
    private Long userId;

    @NotEmpty(message = "Order must contain at least one item")
    @Valid
    private List<OrderItemRequest> items;

    @NotBlank(message = "Shipping address is required")
    private String shippingAddress;

    // Getters and Setters
    public Long getUserId() { return userId; }
    public void setUserId(Long userId) { this.userId = userId; }

    public List<OrderItemRequest> getItems() { return items; }
    public void setItems(List<OrderItemRequest> items) { this.items = items; }

    public String getShippingAddress() { return shippingAddress; }
    public void setShippingAddress(String shippingAddress) { this.shippingAddress = shippingAddress; }
}
```

**Explanation:**  
DTO for creating an order. Includes validation for user ID, items, and shipping address.

---

## 6. OrderItemRequest DTO

```java
package com.ecommerce.orderservice.dto;

import jakarta.validation.constraints.*;

public class OrderItemRequest {

    @NotNull(message = "Product ID is required")
    private Long productId;

    @Min(value = 1, message = "Quantity must be at least 1")
    private Integer quantity;

    @NotNull(message = "Unit price is required")
    @Positive(message = "Unit price must be positive")
    private Double unitPrice;

    // Getters and Setters
    public Long getProductId() { return productId; }
    public void setProductId(Long productId) { this.productId = productId; }

    public Integer getQuantity() { return quantity; }
    public void setQuantity(Integer quantity) { this.quantity = quantity; }

    public Double getUnitPrice() { return unitPrice; }
    public void setUnitPrice(Double unitPrice) { this.unitPrice = unitPrice; }
}
```

**Explanation:**  
DTO for individual order item input with validation.

---

## 7. OrderResponse DTO

```java
package com.ecommerce.orderservice.dto;

import java.time.LocalDateTime;
import java.util.List;

public class OrderResponse {

    private Long id;
    private Long userId;
    private List<OrderItemResponse> items;
    private Double totalAmount;
    private OrderStatus status;
    private PaymentStatus paymentStatus;
    private String shippingAddress;
    private LocalDateTime createdAt;
    private LocalDateTime updatedAt;

    // Getters and Setters (omitted for brevity)
}
```

**Explanation:**  
DTO returned to clients after order operations.

---

## 8. OrderItemResponse DTO

```java
package com.ecommerce.orderservice.dto;

public class OrderItemResponse {

    private Long id;
    private Long productId;
    private Integer quantity;
    private Double unitPrice;
    private Double totalPrice;

    // Getters and Setters (omitted for brevity)
}
```

**Explanation:**  
DTO representing an order item in responses.

---

## 9. Custom Exception

```java
package com.ecommerce.orderservice.exception;

import org.springframework.http.HttpStatus;
import org.springframework.web.server.ResponseStatusException;

public class OrderNotFoundException extends ResponseStatusException {

    public OrderNotFoundException(String message) {
        super(HttpStatus.NOT_FOUND, message);
    }
}
```

**Explanation:**  
Custom exception thrown when an order is not found. Returns HTTP 404.

---

## 10. Order Repository

```java
package com.ecommerce.orderservice.repository;

import com.ecommerce.orderservice.model.Order;
import org.springframework.data.jpa.repository.JpaRepository;
import org.springframework.stereotype.Repository;

import java.util.List;

@Repository
public interface OrderRepository extends JpaRepository<Order, Long> {

    List<Order> findByUserId(Long userId);
    boolean existsById(Long id);
}
```

**Explanation:**  
Spring Data JPA repository for Order entity. Includes custom query to find orders by user.

---

## 11. Order Service Interface

```java
package com.ecommerce.orderservice.service;

import com.ecommerce.orderservice.dto.OrderRequest;
import com.ecommerce.orderservice.dto.OrderResponse;
import java.util.List;

public interface OrderService {

    OrderResponse createOrder(OrderRequest request);
    OrderResponse getOrderById(Long orderId);
    List<OrderResponse> getOrdersByUserId(Long userId);
    OrderResponse updateOrderStatus(Long orderId, OrderStatus status);
    OrderResponse cancelOrder(Long orderId);
}
```

**Explanation:**  
Defines the business contract for order operations.

---

## 12. Order Service Implementation

```java
package com.ecommerce.orderservice.service;

import com.ecommerce.orderservice.dto.OrderRequest;
import com.ecommerce.orderservice.dto.OrderResponse;
import com.ecommerce.orderservice.exception.OrderNotFoundException;
import com.ecommerce.orderservice.model.Order;
import com.ecommerce.orderservice.model.OrderItem;
import com.ecommerce.orderservice.model.OrderStatus;
import com.ecommerce.orderservice.model.PaymentStatus;
import com.ecommerce.orderservice.repository.OrderRepository;
import lombok.RequiredArgsConstructor;
import lombok.extern.slf4j.Slf4j;
import org.springframework.stereotype.Service;
import org.springframework.transaction.annotation.Transactional;
import java.util.List;
import java.util.stream.Collectors;

@Service
@RequiredArgsConstructor
@Slf4j
@Transactional
public class OrderServiceImpl implements OrderService {

    private final OrderRepository orderRepository;

    @Override
    public OrderResponse createOrder(OrderRequest request) {
        log.info("Creating order for user: {}", request.getUserId());

        Order order = Order.builder()
                .userId(request.getUserId())
                .shippingAddress(request.getShippingAddress())
                .status(OrderStatus.PENDING)
                .paymentStatus(PaymentStatus.PENDING)
                .build();

        double totalAmount = 0;
        List<OrderItem> orderItems = request.getItems().stream().map(itemRequest -> {
            double itemTotal = itemRequest.getQuantity() * itemRequest.getUnitPrice();
            totalAmount += itemTotal;

            return OrderItem.builder()
                    .productId(itemRequest.getProductId())
                    .quantity(itemRequest.getQuantity())
                    .unitPrice(itemRequest.getUnitPrice())
                    .totalPrice(itemTotal)
                    .order(order)
                    .build();
        }).collect(Collectors.toList());

        order.setOrderItems(orderItems);
        order.setTotalAmount(totalAmount);

        Order savedOrder = orderRepository.save(order);
        log.info("Order created successfully with ID: {}", savedOrder.getId());

        return mapToResponse(savedOrder);
    }

    @Override
    public OrderResponse getOrderById(Long orderId) {
        log.info("Fetching order by ID: {}", orderId);
        Order order = orderRepository.findById(orderId)
                .orElseThrow(() -> new OrderNotFoundException("Order not found with ID: " + orderId));
        return mapToResponse(order);
    }

    @Override
    public List<OrderResponse> getOrdersByUserId(Long userId) {
        log.info("Fetching orders for user: {}", userId);
        List<Order> orders = orderRepository.findByUserId(userId);
        return orders.stream().map(this::mapToResponse).collect(Collectors.toList());
    }

    @Override
    public OrderResponse updateOrderStatus(Long orderId, OrderStatus status) {
        log.info("Updating order ID: {} to status: {}", orderId, status);
        Order order = orderRepository.findById(orderId)
                .orElseThrow(() -> new OrderNotFoundException("Order not found with ID: " + orderId));

        order.setStatus(status);
        Order updatedOrder = orderRepository.save(order);
        return mapToResponse(updatedOrder);
    }

    @Override
    public OrderResponse cancelOrder(Long orderId) {
        log.info("Cancelling order ID: {}", orderId);
        Order order = orderRepository.findById(orderId)
                .orElseThrow(() -> new OrderNotFoundException("Order not found with ID: " + orderId));

        if (order.getStatus() == OrderStatus.DELIVERED) {
            throw new IllegalStateException("Cannot cancel a delivered order");
        }

        order.setStatus(OrderStatus.CANCELLED);
        Order cancelledOrder = orderRepository.save(order);
        return mapToResponse(cancelledOrder);
    }

    private OrderResponse mapToResponse(Order order) {
        OrderResponse response = new OrderResponse();
        response.setId(order.getId());
        response.setUserId(order.getUserId());
        response.setTotalAmount(order.getTotalAmount());
        response.setStatus(order.getStatus());
        response.setPaymentStatus(order.getPaymentStatus());
        response.setShippingAddress(order.getShippingAddress());
        response.setCreatedAt(order.getCreatedAt());
        response.setUpdatedAt(order.getUpdatedAt());

        List<OrderItemResponse> itemResponses = order.getOrderItems().stream().map(item -> {
            OrderItemResponse itemResponse = new OrderItemResponse();
            itemResponse.setId(item.getId());
            itemResponse.setProductId(item.getProductId());
            itemResponse.setQuantity(item.getQuantity());
            itemResponse.setUnitPrice(item.getUnitPrice());
            itemResponse.setTotalPrice(item.getTotalPrice());
            return itemResponse;
        }).collect(Collectors.toList());

        response.setItems(itemResponses);
        return response;
    }
}
```

**Explanation:**  
- `@Service`: Marks as Spring service bean.
- `@Transactional`: Ensures atomic operations.
- `createOrder()`: Builds order and items, calculates total, saves to DB.
- `getOrderById()`, `getOrdersByUserId()`: Fetch orders with exception handling.
- `updateOrderStatus()`, `cancelOrder()`: Modify order status with validation.
- `mapToResponse()`: Converts entity to DTO.

---

## 13. Order Controller

```java
package com.ecommerce.orderservice.controller;

import com.ecommerce.orderservice.dto.OrderRequest;
import com.ecommerce.orderservice.dto.OrderResponse;
import com.ecommerce.orderservice.model.OrderStatus;
import com.ecommerce.orderservice.service.OrderService;
import lombok.RequiredArgsConstructor;
import lombok.extern.slf4j.Slf4j;
import org.springframework.http.ResponseEntity;
import org.springframework.web.bind.*;
import java.util.List;

@RestController
@RequestMapping("/api/orders")
@RequiredArgsConstructor
@Slf4j
public class OrderController {

    private final OrderService orderService;

    @PostMapping
    public ResponseEntity<OrderResponse> createOrder(@RequestBody @Valid OrderRequest request) {
        log.info("Received request to create order");
        OrderResponse response = orderService.createOrder(request);
        return ResponseEntity.ok(response);
    }

    @GetMapping("/{orderId}")
    public ResponseEntity<OrderResponse> getOrderById(@PathVariable Long orderId) {
        OrderResponse response = orderService.getOrderById(orderId);
        return ResponseEntity.ok(response);
    }

    @GetMapping("/user/{userId}")
    public ResponseEntity<List<OrderResponse>> getOrdersByUserId(@PathVariable Long userId) {
        List<OrderResponse> responses = orderService.getOrdersByUserId(userId);
        return ResponseEntity.ok(responses);
    }

    @PutMapping("/{orderId}/status")
    public ResponseEntity<OrderResponse> updateOrderStatus(
            @PathVariable Long orderId,
            @RequestParam OrderStatus status) {
        OrderResponse response = orderService.updateOrderStatus(orderId, status);
        return ResponseEntity.ok(response);
    }

    @DeleteMapping("/{orderId}")
    public ResponseEntity<OrderResponse> cancelOrder(@PathVariable Long orderId) {
        OrderResponse response = orderService.cancelOrder(orderId);
        return ResponseEntity.ok(response);
    }
}
```

**Explanation:**  
- REST endpoints for order operations.
- `@Valid` ensures request validation.
- Proper HTTP methods and status codes.

---

## 14. Global Exception Handler

```java
package com.ecommerce.orderservice.exception;

import org.springframework.http.ResponseEntity;
import org.springframework.web.bind.annotation.ControllerAdvice;
import org.springframework.web.bind.annotation.ExceptionHandler;

@ControllerAdvice
public class GlobalExceptionHandler {

    @ExceptionHandler(OrderNotFoundException.class)
    public ResponseEntity<String> handleOrderNotFound(OrderNotFoundException ex) {
        return ResponseEntity.status(404).body(ex.getMessage());
    }

    @ExceptionHandler(IllegalStateException.class)
    public ResponseEntity<String> handleIllegalState(IllegalStateException ex) {
        return ResponseEntity.status(400).body(ex.getMessage());
    }

    @ExceptionHandler(Exception.class)
    public ResponseEntity<String> handleGeneralException(Exception ex) {
        return ResponseEntity.status(500).body("Internal server error: " + ex.getMessage());
    }
}
```

**Explanation:**  
Centralized exception handling for clean error responses.

---

## 15. application.properties (Example)

```properties
spring.application.name=order-service
spring.datasource.url=jdbc:mysql://localhost:3306/order_db
spring.datasource.username=root
spring.datasource.password=password
spring.jpa.hibernate.ddl-auto=update
spring.jpa.show-sql=true
server.port=8082
```

**Explanation:**  
Database and server configuration for the Order Service.

---

## ✅ Features Summary

- Full CRUD for orders
- Order item management with price calculation
- Status and payment tracking
- User-specific order retrieval
- Input validation and exception handling
- Transactional integrity
- RESTful API design

This Order Service is ready to be integrated into a larger e-commerce ecosystem (e.g., with Product Service, Payment Service, and Notification Service via messaging).